In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
import numpy as np
import time
import logging
import sys
import json
from datetime import datetime

# --- 配置与环境设置 ---
def setup_logging(log_file='training_mlp_regression.log'):
    log_formatter = logging.Formatter("%(asctime)s [%(levelname)-5.5s]  %(message)s")
    root_logger = logging.getLogger(); root_logger.handlers = []; root_logger.setLevel(logging.INFO)
    file_handler = logging.FileHandler(log_file, mode='w'); file_handler.setFormatter(log_formatter); root_logger.addHandler(file_handler)
    console_handler = logging.StreamHandler(sys.stdout); console_handler.setFormatter(log_formatter); root_logger.addHandler(console_handler)

torch.manual_seed(42); np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- 数据加载器 ---
def get_regression_data_loaders(batch_size=128, test_size=0.3):
    """准备在线新闻流行度回归数据集"""
    logging.info("正在准备在线新闻流行度回归数据集...")
    
    import pandas as pd
    df = pd.read_csv('/kaggle/input/online-news-popularity/OnlineNewsPopularity.csv')
    
    # 删除无用和共线性特征
    cols_to_drop = ['url', ' timedelta', 
                   ' abs_title_sentiment_polarity', ' LDA_04', ' is_weekend',
                   ' rate_positive_words', ' abs_title_subjectivity', 
                   ' rate_negative_words']
    df = df.drop(cols_to_drop, axis=1)
    
    # 特征和目标变量
    X = df.drop([' shares'], axis=1)  # 特征矩阵
    y = df[' shares']  # 目标变量（文章分享次数）
    
    # 数据标准化
    from sklearn.preprocessing import StandardScaler
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    
    # 对目标变量取对数并标准化（因为分享次数通常呈长尾分布）
    y = np.log1p(y)  # 使用log(1+y)转换
    y = (y - y.mean()) / y.std()  # 标准化目标变量
    
    # 划分训练测试集
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    
    # 转换为PyTorch张量
    X_train = torch.FloatTensor(X_train)
    y_train = torch.FloatTensor(y_train.values)  
    X_test = torch.FloatTensor(X_test)
    y_test = torch.FloatTensor(y_test.values)   
   
    # 创建DataLoader
    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
    
    trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    logging.info("在线新闻流行度回归数据集准备完毕。")
    logging.info(f"训练样本数: {len(train_dataset)}, 测试样本数: {len(test_dataset)}")
    logging.info(f"特征维度: {X_train.shape[1]}, 目标维度: 1")
    
    return trainloader, testloader, X_train.shape[1]

# --- MLP模型定义 ---
class MLPRegressor(nn.Module):
    """回归任务专用的MLP模型"""
    def __init__(self, input_dim, hidden_dims=[256, 128], dropout_rate=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        
        # 动态构建隐藏层
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.BatchNorm1d(hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = hidden_dim
        
        # 输出层
        layers.append(nn.Linear(prev_dim, 1))
        
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(x).squeeze()  # 去掉多余的维度

# --- 训练函数 ---
def train_model(model, trainloader, testloader, hparams, log_filepath, model_save_path):
    criterion = nn.MSELoss()  # 回归任务使用MSE损失
    best_test_loss = float('inf')
    
    log_data = {
        "experiment_timestamp": datetime.now().isoformat(),
        "hyperparameters": hparams,
        "training_loss_per_epoch": [],
        "test_loss_per_epoch": []
    }

    logging.info("="*30)
    logging.info("  启动MLP回归模型训练流程")
    logging.info("="*30)
    
    optimizer = optim.Adam(model.parameters(), 
                         lr=hparams['learning_rate'], 
                         weight_decay=hparams['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=hparams['epochs'])

    for epoch in range(hparams['epochs']):
        # 训练阶段
        model.train()
        running_loss = 0.0
        for inputs, targets in trainloader:
            inputs, targets = inputs.to(device), targets.float().to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        
        train_epoch_loss = running_loss / len(trainloader)
        log_data["training_loss_per_epoch"].append(train_epoch_loss)
        
        # 测试阶段
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(device), targets.float().to(device)
                outputs = model(inputs)
                test_loss += criterion(outputs, targets).item()
        
        test_epoch_loss = test_loss / len(testloader)
        log_data["test_loss_per_epoch"].append(test_epoch_loss)
        
        scheduler.step()
        
        logging.info(f"Epoch {epoch+1}/{hparams['epochs']} | 训练MSE损失: {train_epoch_loss:.4f} | 测试MSE损失: {test_epoch_loss:.4f}")
        
        # 保存最佳模型
        if test_epoch_loss < best_test_loss:
            best_test_loss = test_epoch_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'best_loss': best_test_loss,
                'hyperparameters': hparams
            }, model_save_path)
            logging.info(f"*** 新的最佳MSE损失: {best_test_loss:.4f} ***")
        
        with open(log_filepath, 'w') as f:
            json.dump(log_data, f, indent=4)
        
        if best_test_loss < hparams['loss_target']:
            logging.info(f"目标损失 {hparams['loss_target']} 已达到。")
            break

    logging.info(f"训练流程结束。最佳测试MSE损失: {best_test_loss:.4f}")
    return best_test_loss

# --- 主程序 ---
if __name__ == '__main__':
    # 文件与超参数定义
    LOG_FILE = 'training_mlp_regression.log'
    JSON_LOG_FILE = 'training_mlp_regression_log.json'
    MODEL_SAVE_PATH = 'regression_mlp.pth'
    
    HYPERPARAMETERS = {
        "epochs": 100,          # 训练轮数
        "loss_target": 0.1,     # MSE目标值
        "hidden_dims": [256, 128],  # MLP隐藏层维度
        "dropout_rate": 0.2,    # Dropout率
        "learning_rate": 1e-3,  # 学习率
        "weight_decay": 1e-5,   # 权重衰减
    }

    setup_logging(LOG_FILE)
    logging.info(f"回归实验超参数: {json.dumps(HYPERPARAMETERS, indent=4)}")

    trainloader, testloader, input_dim = get_regression_data_loaders()
    
    # 创建MLP模型
    model = MLPRegressor(
        input_dim=input_dim,
        hidden_dims=HYPERPARAMETERS['hidden_dims'],
        dropout_rate=HYPERPARAMETERS['dropout_rate']
    ).to(device)

    start_time = time.time()
    best_loss = train_model(
        model, trainloader, testloader,
        hparams=HYPERPARAMETERS,
        log_filepath=JSON_LOG_FILE,
        model_save_path=MODEL_SAVE_PATH
    )
    
    total_time = time.time() - start_time
    logging.info(f"总训练时间: {total_time:.2f} 秒")
    
    # 加载并验证保存的最佳模型
    logging.info("\n" + "="*30)
    logging.info("  开始验证已保存的最佳回归模型")
    logging.info("="*30)
    
    verification_model = MLPRegressor(
        input_dim=input_dim,
        hidden_dims=HYPERPARAMETERS['hidden_dims'],
        dropout_rate=HYPERPARAMETERS['dropout_rate']
    ).to(device)

    checkpoint = torch.load(MODEL_SAVE_PATH)
    verification_model.load_state_dict(checkpoint['model_state_dict'])
    verification_model.eval()

    logging.info(f"成功加载模型。保存的MSE损失: {checkpoint['best_loss']:.4f}")

    total_params = sum(p.numel() for p in verification_model.parameters() if p.requires_grad)
    logging.info(f"模型总参数数量: {total_params}")

    test_loss = 0.0
    criterion = nn.MSELoss()
    with torch.no_grad():
        for inputs, targets in testloader:
            inputs, targets = inputs.to(device), targets.float().to(device)
            outputs = verification_model(inputs)
            test_loss += criterion(outputs, targets).item()
    
    avg_test_loss = test_loss / len(testloader)
    logging.info(f"加载后的模型在测试集上的MSE损失: {avg_test_loss:.4f}")
    
    assert abs(avg_test_loss - checkpoint['best_loss']) < 1e-3, "验证失败！"
    logging.info("验证成功！回归模型可以被正确保存和复用。")

2025-08-14 10:08:09,590 [INFO ]  回归实验超参数: {
    "epochs": 100,
    "loss_target": 0.1,
    "hidden_dims": [
        256,
        128
    ],
    "dropout_rate": 0.2,
    "learning_rate": 0.001,
    "weight_decay": 1e-05
}
2025-08-14 10:08:09,591 [INFO ]  正在准备在线新闻流行度回归数据集...
2025-08-14 10:08:09,746 [INFO ]  NumExpr defaulting to 4 threads.
2025-08-14 10:08:11,583 [INFO ]  在线新闻流行度回归数据集准备完毕。
2025-08-14 10:08:11,584 [INFO ]  训练样本数: 27750, 测试样本数: 11894
2025-08-14 10:08:11,584 [INFO ]  特征维度: 52, 目标维度: 1
2025-08-14 10:08:11,612 [INFO ]  ==============================
2025-08-14 10:08:11,613 [INFO ]    启动MLP回归模型训练流程
2025-08-14 10:08:11,613 [INFO ]  ==============================
2025-08-14 10:08:16,000 [INFO ]  Epoch 1/100 | 训练MSE损失: 0.9193 | 测试MSE损失: 0.8748
2025-08-14 10:08:16,004 [INFO ]  *** 新的最佳MSE损失: 0.8748 ***
2025-08-14 10:08:17,040 [INFO ]  Epoch 2/100 | 训练MSE损失: 0.8657 | 测试MSE损失: 0.8698
2025-08-14 10:08:17,043 [INFO ]  *** 新的最佳MSE损失: 0.8698 ***
2025-08-14 10:08:18,062 [INFO ]  Epoch 3/